In [14]:
import copy
import torch
import torch.nn as nn 

from torch.utils.data import Dataset, DataLoader

from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor



In [15]:
device = 'cuda'

In [16]:
class Net(nn.Module):
       def __init__(self, inDim, featureDim, outDim):
              super(Net, self).__init__()
              self.classifier = nn.Sequential(

                     nn.Linear(inDim, featureDim, bias= True, device=device),
                     nn.ReLU(),
                     nn.Linear(featureDim, outDim, bias=True, device=device)
              ) 

       def forward(self, x):
              return self.classifier(x)


In [22]:
inDim , featureDim, outDim = 784, 256, 10

model = Net(inDim, featureDim, outDim)

lossFn = nn.CrossEntropyLoss()

#  loading the model previously saved as mnist.pt


## Specifically load the mnist.pt, the save model and then load the stateDict in the load_state_dict method
stateDict = torch.load("/home/bourbaki/college/7thSem/project7thSem/DNN/mnist.pt", map_location=device)

model.load_state_dict(state_dict= stateDict)


# now we can access the parameters of the model like the weights and the biases of the model 
opt_state_dict = copy.deepcopy(model.state_dict())


for paramTensor in opt_state_dict:
       print(paramTensor, "\t", opt_state_dict[paramTensor].size())




classifier.0.weight 	 torch.Size([256, 784])
classifier.0.bias 	 torch.Size([256])
classifier.2.weight 	 torch.Size([10, 256])
classifier.2.bias 	 torch.Size([10])


In [18]:
# creating a random wieghted model

model_rand = Net(inDim, featureDim, outDim)
rand_state_dict = copy.deepcopy(model_rand.state_dict())

In [19]:
# create a new stae_dict for interplated parameters

test_model = Net(inDim, featureDim, outDim)
test_state_dict = copy.deepcopy(test_model.state_dict())

alpha = 0.2
beta = 1.0 - alpha

for p in opt_state_dict:
       test_state_dict[p] = (opt_state_dict[p] * beta + rand_state_dict[p] * beta)





In [20]:
train_dataset = MNIST(".", train= True, download= True, transform=ToTensor())
test_dataset = MNIST(".", train = False, download = True, transform= ToTensor())

trainLoader = DataLoader(train_dataset, batch_size= 64, shuffle = True)
testLoader =  DataLoader(test_dataset, batch_size= 64,  shuffle = False)

100%|██████████| 9.91M/9.91M [00:33<00:00, 293kB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 94.2kB/s]
100%|██████████| 1.65M/1.65M [00:04<00:00, 375kB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.2MB/s]


In [26]:
def inference(testLoader, model, lossFn):
       rinningLoss = 0.0
       with torch.no_grad():
              for inputs, labels in testLoader:
                     inputs = inputs.to_device()
                     outputs = model(inputs)
                     loss = lossFn(outputs, labels)
              runningLoss /= len(testLoader)
              return runningLoss
                     

In [27]:
results = []
for alpha in torch.arange(-2.0, 2, 0.05):
       beta = 1.0 -alpha


       for p in opt_state_dict:
              test_state_dict[p] = (opt_state_dict[p]*beta + test_state_dict[p]* alpha)


       model.load_state_dict(test_state_dict)

       loss = inference(trainLoader, model, lossFn)
       results.append(loss.item())

AttributeError: 'Tensor' object has no attribute 'to_device'